In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_csv('../data/raw/call_center.csv')
df.sample(5, random_state = 42)

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count
575,C0576,1/17/2026,12:05:20,912000945,Order Status,A12,Billing,Phone,Answered,100,269.0,76.0,21.0,2.0,Yes,0
259,C0260,1/9/2026,16:52:00,912000159,Refund Request,A11,Billing,Phone,Answered,73,207.0,10.0,64.0,5.0,Yes,0
388,C0389,1/11/2026,18:49:51,912000874,Refund Request,A01,Billing,Phone,Answered,81,175.0,77.0,54.0,4.0,Yes,1
495,C0496,1/15/2026,14:27:05,912000019,Product Inquiry,A08,Sales,Phone,Answered,82,179.0,68.0,53.0,3.0,Yes,0
70,C0071,1/3/2026,19:23:26,912001332,Complaint,A09,Support,Callback,Answered,167,343.0,15.0,53.0,1.0,Yes,0


## Inspect Data

In [3]:
df.columns

Index(['call_id', 'call_date', 'start_time', 'customer_phone', 'call_reason',
       'agent_id', 'queue', 'channel', 'status', 'wait_seconds',
       'handle_seconds', 'hold_seconds', 'after_call_work_seconds',
       'csat_score', 'first_contact_resolution', 'transfer_count'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 968 entries, 0 to 967
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   call_id                   968 non-null    object 
 1   call_date                 968 non-null    object 
 2   start_time                968 non-null    object 
 3   customer_phone            968 non-null    object 
 4   call_reason               968 non-null    object 
 5   agent_id                  964 non-null    object 
 6   queue                     968 non-null    object 
 7   channel                   968 non-null    object 
 8   status                    968 non-null    object 
 9   wait_seconds              968 non-null    int64  
 10  handle_seconds            848 non-null    float64
 11  hold_seconds              846 non-null    float64
 12  after_call_work_seconds   846 non-null    float64
 13  csat_score                846 non-null    float64
 14  first_cont

In [5]:
df.describe()

,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,transfer_count
count,968.000000,848.000000,846.000000,846.000000,846.000000,968.000000
mean,98.318182,306.278302,42.968085,47.784870,3.591017,0.373967
std,52.009426,110.535139,21.625651,15.830868,1.174931,0.632312
min,10.000000,-120.000000,5.000000,20.000000,1.000000,0.000000
25%,56.000000,218.000000,25.000000,35.000000,3.000000,0.000000
50%,88.000000,284.000000,42.000000,48.000000,4.000000,0.000000
75%,139.000000,390.000000,62.000000,62.000000,5.000000,1.000000
max,257.000000,663.000000,80.000000,75.000000,6.000000,2.000000


In [6]:
df.isna().sum()

call_id                       0
call_date                     0
start_time                    0
customer_phone                0
call_reason                   0
agent_id                      4
queue                         0
channel                       0
status                        0
wait_seconds                  0
handle_seconds              120
hold_seconds                122
after_call_work_seconds     122
csat_score                  122
first_contact_resolution    122
transfer_count                0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(2)

In [8]:
df.nunique()

call_id                     965
call_date                    29
start_time                  950
customer_phone              966
call_reason                   8
agent_id                     12
queue                         3
channel                       2
status                        3
wait_seconds                199
handle_seconds              336
hold_seconds                 76
after_call_work_seconds      56
csat_score                    6
first_contact_resolution      2
transfer_count                3
dtype: int64

### check inconsistence data

In [9]:
df[df['status']=='Answered'].isna().sum()

call_id                     0
call_date                   0
start_time                  0
customer_phone              0
call_reason                 0
agent_id                    4
queue                       0
channel                     0
status                      0
wait_seconds                0
handle_seconds              0
hold_seconds                0
after_call_work_seconds     0
csat_score                  0
first_contact_resolution    0
transfer_count              0
dtype: int64

In [10]:
df[df['agent_id'].isna()]

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count
16,C0017,1/1/2026,8:40:41,912000706,Refund Request,NaN,Support,Phone,Answered,106,344.0,24.0,41.0,5.0,Yes,0
195,C0196,1/7/2026,18:24:33,0912000870,Product Inquiry,NaN,Sales,Phone,Answered,42,244.0,77.0,49.0,3.0,No,0
272,C0273,1/9/2026,14:50:14,"+98912001153, +98912004453",Billing Issue,NaN,Support,Phone,Answered,115,412.0,73.0,20.0,2.0,Yes,0
863,C0867,1/26/2026,19:37:24,912000212,Order Status,NaN,Sales,Phone,Answered,86,213.0,31.0,41.0,5.0,Yes,0


In [11]:
df[df['status']=='Abandoned'].isna().sum()

call_id                       0
call_date                     0
start_time                    0
customer_phone                0
call_reason                   0
agent_id                      0
queue                         0
channel                       0
status                        0
wait_seconds                  0
handle_seconds              120
hold_seconds                122
after_call_work_seconds     122
csat_score                  122
first_contact_resolution    122
transfer_count                0
dtype: int64

In [12]:
df[(df['status'] == 'Abandoned') & ~(df['handle_seconds'].isna())]

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count
15,C0016,1/1/2026,15:10:23,912000445,Account Access,A07,Support,Phone,Abandoned,185,40.0,NaN,NaN,NaN,NaN,0
490,C0491,1/15/2026,18:06:53,912000166,Complaint,A08,Support,Phone,Abandoned,179,25.0,NaN,NaN,NaN,NaN,0


In [13]:
df[df.duplicated(keep = False)]

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count
49,C0050,1/2/2026,15:05:25,912000934,Technical Problem,A04,Support,Callback,Answered,87,352.0,68.0,58.0,3.0,Yes,1
199,C0200,1/7/2026,19:29:46,912001300,Product Inquiry,A07,Support,Phone,Transferred,107,652.0,50.0,68.0,5.0,No,0
965,C0050,1/2/2026,15:05:25,912000934,Technical Problem,A04,Support,Callback,Answered,87,352.0,68.0,58.0,3.0,Yes,1
966,C0200,1/7/2026,19:29:46,912001300,Product Inquiry,A07,Support,Phone,Transferred,107,652.0,50.0,68.0,5.0,No,0


In [14]:
# check for unreasonable values
for c in df.columns:
    if df[c].nunique()  < 30:
        print(f"{c} : ")
        print(df[c].unique())
        print()
    

call_date : 
['1/1/2026' '1/2/2026' '1/3/2026' '1/4/2026' '1/5/2026' '1/6/2026'
 '1/7/2026' '1/8/2026' '1/9/2026' '1/10/2026' '1/11/2026' '1/12/2026'
 '1/13/2026' '1/14/2026' '1/15/2026' '1/16/2026' '1/17/2026' '1/18/2026'
 '1/19/2026' '1/20/2026' '1/21/2026' '1/22/2026' '1/23/2026' '1/24/2026'
 '1/25/2026' '1/26/2026' '1/27/2026' '1/28/2026' '1/29/2026']

call_reason : 
['Account Access' 'Order Status' 'Technical Problem' 'Product Inquiry'
 'Refund Request' 'Complaint' 'Billing Issue' 'Subscription Change']

agent_id : 
['A06' 'A07' 'A12' 'A10' 'A02' 'A09' 'A01' 'A04' nan 'A03' 'A11' 'A05'
 'A08']

queue : 
['Billing' 'Sales' 'Support']

channel : 
['Phone' 'Callback']

status : 
['Answered' 'Abandoned' 'Transferred']

csat_score : 
[ 3. nan  4.  5.  2.  1.  6.]

first_contact_resolution : 
['No' nan 'Yes']

transfer_count : 
[0 2 1]



In [15]:
df[df['csat_score'] > 5]

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count
452,C0453,1/13/2026,18:07:24,912001079,Account Access,A08,Support,Callback,Answered,89,422.0,43.0,65.0,6.0,Yes,0
847,C0851,1/26/2026,9:51:33,912000400,Product Inquiry,A06,Support,Phone,Answered,179,431.0,40.0,71.0,6.0,Yes,1
934,C0938,1/29/2026,9:11:32,912000476,Account Access,A02,Billing,Phone,Answered,46,175.0,67.0,67.0,6.0,Yes,0


In [16]:
# check time
time_part = pd.to_datetime(df['start_time'], format='%H:%M:%S', errors='coerce').dt.time
time_part.isna().sum()

np.int64(0)

In [17]:
#Determining the hour range
print(time_part.min())
print(time_part.max())

08:00:40
19:59:57


## Convert Data type

In [18]:
toStr_col = ['call_id','customer_phone']
for col in toStr_col:
    df[col] = df[col].astype('string').str.strip()

toCat_col = ['call_reason', 'agent_id', 'queue', 'channel', 'status', 'first_contact_resolution']
for col in toCat_col:
    df[col] = df[col].astype('category')

toInt_col = ['handle_seconds','hold_seconds','after_call_work_seconds','csat_score','transfer_count']
for col in toInt_col:
    df[col] = df[col].astype('Int64')

df['call_date'] = pd.to_datetime(df['call_date'],format='%m/%d/%Y')
df['start_time'] = pd.to_timedelta(df['start_time'])

# Create new columns for analysis purposes
df['call_datetime'] = df['call_date'] + df['start_time']
df['hour'] = df['call_datetime'].dt.hour
df['week_day'] = df['call_datetime'].dt.day_name()          # Monday, Tuesday...
df['day'] = df['call_datetime'].dt.day

In [20]:
df[['call_datetime', 'hour', 'week_day','day']].sample(2)

,call_datetime,hour,week_day,day
398,2026-01-12 16:13:09,16,Monday,12
240,2026-01-08 16:00:35,16,Thursday,8


In [21]:
df.describe()

,call_date,start_time,wait_seconds,handle_seconds,hold_seconds,after_call_work_seconds,csat_score,transfer_count,call_datetime,hour,day
count,968,968,968.000000,848.0,846.0,846.0,846.0,968.0,968,968.000000,968.000000
mean,2026-01-14 17:25:47.107437824,0 days 14:26:54.973140495,98.318182,306.278302,42.968085,47.78487,3.591017,0.373967,2026-01-15 07:52:42.080578560,13.962810,14.726240
min,2026-01-01 00:00:00,0 days 08:00:40,10.000000,-120.0,5.0,20.0,1.0,0.0,2026-01-01 08:32:04,8.000000,1.000000
25%,2026-01-08 00:00:00,0 days 11:19:22,56.000000,218.0,25.0,35.0,3.0,0.0,2026-01-08 15:33:24.500000,11.000000,8.000000
50%,2026-01-14 00:00:00,0 days 15:10:14.500000,88.000000,284.0,42.0,48.0,4.0,0.0,2026-01-14 17:47:19,15.000000,14.000000
75%,2026-01-22 00:00:00,0 days 17:05:59,139.000000,390.0,62.0,62.0,5.0,1.0,2026-01-22 11:14:28.500000,17.000000,22.000000
max,2026-01-29 00:00:00,0 days 19:59:57,257.000000,663.0,80.0,75.0,6.0,2.0,2026-01-29 18:55:32,19.000000,29.000000
std,NaN,0 days 03:18:00.663732259,52.009426,110.535139,21.625651,15.830868,1.174931,0.632312,NaN,3.294987,8.273533


In [22]:
def assign_shift(hour):
    if hour < 14:
        return 'Morning'
    else:
        return 'Afternoon'
        


In [23]:
df['shift_time'] = df['hour'].apply(assign_shift)
df.tail(2)

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,...,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count,call_datetime,hour,week_day,day,shift_time
966,C0200,2026-01-07,0 days 19:29:46,912001300,Product Inquiry,A07,Support,Phone,Transferred,107,...,50,68,5,No,0,2026-01-07 19:29:46,19,Wednesday,7,Afternoon
967,C0500,2026-01-15,0 days 16:29:48,+98912000206,Subscription Change,A05,Support,Phone,Answered,173,...,32,31,2,No,0,2026-01-15 16:29:48,16,Thursday,15,Afternoon


## Cleaning Data

In [24]:
# remove missing agent_id records
df.dropna(subset =['agent_id'], inplace = True)

# Remove duplicated data
df.drop_duplicates(inplace = True)

#remove negative time
df = df[(df['status'] == 'Abandoned') |(df.handle_seconds > 0)]


In [25]:
# fill wrong score with avg of agent score
mask = df['csat_score'].between(1,5)
avg_agent = df[mask].groupby('agent_id', observed = True)['csat_score'].mean().round().clip(1, 5).astype('Int64')
df.loc[~mask,'csat_score'] = df.loc[~mask, 'agent_id'].map(avg_agent)

In [26]:
print(avg_agent)
# check
df[df.csat_score>5]

agent_id
A01    4
A02    4
A03    4
A04    4
A05    3
A06    3
A07    4
A08    4
A09    4
A10    4
A11    4
A12    4
Name: csat_score, dtype: Int64


,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,...,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count,call_datetime,hour,week_day,day,shift_time


In [27]:
# check csat score 
df[df.call_id.isin(['C0453','C0851','C0938'])]

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,...,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count,call_datetime,hour,week_day,day,shift_time
452,C0453,2026-01-13,0 days 18:07:24,912001079,Account Access,A08,Support,Callback,Answered,89,...,43,65,4,Yes,0,2026-01-13 18:07:24,18,Tuesday,13,Afternoon
847,C0851,2026-01-26,0 days 09:51:33,912000400,Product Inquiry,A06,Support,Phone,Answered,179,...,40,71,3,Yes,1,2026-01-26 09:51:33,9,Monday,26,Morning
934,C0938,2026-01-29,0 days 09:11:32,912000476,Account Access,A02,Billing,Phone,Answered,46,...,67,67,4,Yes,0,2026-01-29 09:11:32,9,Thursday,29,Morning


In [28]:
# find incorrect call_id
pattern = r'^C\d{4}$'
invalid_callid = ~(df['call_id'].str.match(pattern, na=False))
df[invalid_callid]

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,...,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count,call_datetime,hour,week_day,day,shift_time


## SAVE Clean Data

In [29]:
from pathlib import Path
# sort before saving
df = df.sort_values(["call_datetime"], ascending = True).reset_index(drop=True)

#df = df.drop(columns=['start_time', 'call_date'], errors='ignore')    #Extra Column

data_clean_dir = Path("../data/preprocess")
data_clean_dir.mkdir(parents=True,exist_ok=True)
df.to_csv(data_clean_dir/"clean_call_center.csv", index = False)

In [ ]:
from kpi import calculate_all_kpis

import pandas as pd


In [ ]:


from kpi import calculate_all_kpis
dfc = pd.read_csv("../data/preprocess/clean_call_center.csv")

In [ ]:
dfc['status'].value_counts()

In [ ]:
# "queue == '' and shift_time == '' and day_name == '' or call_date == ''"

filtered = dfc.query("queue in ['Sales','Support'] and day.between(1, 15)")

results = calculate_all_kpis(filtered)
for key,value in results.items():
    print(f"{key} : {value}")


In [ ]:
dfc.info()